# Bayesian Regression

In [47]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=100, 
    n_features=2, n_informative=2, 
    n_targets=1, 
    bias=0.0, 
    effective_rank=None, 
    tail_strength=0.5, 
    noise=0.0, 
    shuffle=True, 
    coef=False, random_state=42
)
X[:5], y[:5]

(array([[-1.60748323,  0.18463386],
        [-0.26465683,  2.72016917],
        [ 1.46564877, -0.2257763 ],
        [ 1.86577451,  0.47383292],
        [-1.0708925 ,  0.48247242]]),
 array([-127.35915354,  178.28131748,  111.86727647,  198.79808722,
         -58.21718166]))

In [48]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

## Model

Standard linear regression model
$$
y = X w + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)
$$
Bayesian formulation of the model
$$
p(y \mid X, w) = \frac{p(w \mid X, y)p(X, y)}{p(w)} \quad  \Longleftrightarrow \quad p(w \mid X, y) \propto p(y \mid X, w) p(w),
$$
ignoring $p(X, y)$ as a contant wrt. $w$. 

---

Assume prior for the weights 
$$
p(w) = \mathcal{N}(0, \alpha^{-1}I)
$$
where $\alpha$ is a precision parameter.

Posterior over weights
$$
p(w \mid X, y) = \mathcal{N}(\mu, \Sigma),
$$
where
$$
\mu = \frac{1}{\sigma^2} \Sigma X^\top y
$$
$$
\Sigma = (\alpha I + 1 / \sigma^2 X^\top X)^{-1}
$$

---

For a new sample $x_*$ the predictive distribution is
$$
p(y \mid X, w) = \mathcal{N}(\mu_*, \Sigma_*)
$$
where
$$
\mu_* = x_* ^\top \mu
$$
$$
\Sigma_* = x_* ^\top \Sigma x_* + \sigma^2
$$
This gives the mean prediction and uncertainty from variance. Confidence interval is
$$
[ \mu_* \pm 1.96 \sigma_* ]
$$
In $\Sigma_*$, captures epistemic uncertainty in $x_* ^\top \Sigma x_*$ and aleatoric uncertainty in $\sigma^2$.

In [49]:
import numpy as np


class BayesianLinearRegression:
    def __init__(self, alpha=1.0, sigma2=1.0):
        """
        alpha: prior precision (scalar)
        sigma2: noise variance (scalar)
        """
        self.alpha = alpha
        self.sigma2 = sigma2
        self.mu = None       # Posterior mean
        self.Sigma = None    # Posterior covariance

    def fit(self, X, y):
        """
        Fit the model (compute posterior over weights).
        
        X: (N, D) design matrix
        y: (N,) target vector
        """
        N, D = X.shape

        # Compute Sigma = (alpha I + (1/sigma^2) X^T X)^(-1)
        A = self.alpha * np.eye(D) + (1 / self.sigma2) * X.T @ X
        self.Sigma = np.linalg.inv(A)

        # Compute mu = (1/sigma^2) Sigma X^T y
        self.mu = (1 / self.sigma2) * self.Sigma @ X.T @ y

    def predict(self, X_star, return_std=True, return_ci=True):
        """
        Predict for new inputs.
        
        X_star: (N*, D)
        
        Returns:
            mean: (N*,)
            std: (N*,) optional
            ci: (N*, 2) confidence intervals (95%) optional
        """
        if self.mu is None or self.Sigma is None:
            raise ValueError("Model must be fitted before prediction.")

        # Mean: mu_* = X* mu
        mean = X_star @ self.mu

        # Predictive variance: diag(X* Sigma X*^T) + sigma^2
        var = np.sum(X_star @ self.Sigma * X_star, axis=1) + self.sigma2
        std = np.sqrt(var)

        outputs = [mean]

        if return_std:
            outputs.append(std)

        if return_ci:
            ci_lower = mean - 1.96 * std
            ci_upper = mean + 1.96 * std
            ci = np.vstack((ci_lower, ci_upper)).T
            outputs.append(ci)

        return outputs if len(outputs) > 1 else mean

In [53]:
# Fit model
model = BayesianLinearRegression(alpha=1.0, sigma2=0.25)
model.fit(X, y)

# Predict
mean, std, ci = model.predict(X)

print("Predicted mean:\n", mean[:5])
print("\nPredictive std:\n", std[:5])
print("\n95% CI:\n", ci[:5])

Predicted mean:
 [-119.45067352  185.45149744  119.19744907  205.91826727  -50.47577004]

Predictive std:
 [0.50769499 0.51806208 0.50876958 0.51363696 0.50371434]

95% CI:
 [[-120.44575569 -118.45559134]
 [ 184.43609576  186.46689912]
 [ 118.2002607   120.19463744]
 [ 204.91153883  206.9249957 ]
 [ -51.46305015  -49.48848993]]


Posterior covariance
$$
\Sigma = (\alpha I + 1/ \sigma^2 X^\top X)^{-1} 
$$
Posterior mean
$$
\mu = 1 / \sigma^2 \Sigma X ^\top y
$$
Predictive mean
$$
\mu_* = x_* ^\top \mu
$$
Predictive variance
$$
\Sigma_* = x_* ^\top \Sigma x_* + \sigma^2
$$